In [ ]:
# === Google Colab / Local adapter ===
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/Antiplagiarism')
    print('Colab: Drive mounted, working dir set.')
except ImportError:
    print('Local mode.')

In [1]:
import re, time, requests
from pathlib import Path
from urllib.parse import urljoin, unquote
from bs4 import BeautifulSoup

In [2]:
BASE_URL   = "https://official.satbayev.university"
DATA_DIR   = Path("data_PI")
DATA_DIR.mkdir(exist_ok=True)

PI_PAGES = {
    "2025": "/ru/information-telecommunication-technologies/dipomnye-proekty-instituta-avtomatiki-i-informatsionnykh-tekhnologiy/diplomnye-proekty-2025/diplomnye-proekty-pi",
    "2023": "/ru/information-telecommunication-technologies/diplomnye-proekty-aiit-2023/diplomnye-proekty-pi-2023",
    "2022": "/ru/information-telecommunication-technologies/diplomnye-proekty-iaiit-2022/diplomnye-proekty-pi",
    "2021": "/ru/information-telecommunication-technologies/math/diplomnye-proekty-ikiit/diplomnye-proekty-pi",
    "2019": "/ru/diplomnye-proekty-iiitt-2019/diplomnye-proekty-pi",
}

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

print(f"Dir: {DATA_DIR.resolve()}")
print(f"Years: {len(PI_PAGES)}")

Dir: C:\Users\bauir\PyCharmMiscProject\NLP\Antiplagiarism\data_PI
Years: 5


In [ ]:
def get_soup(url: str) -> BeautifulSoup:
    resp = session.get(url, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def sanitize(name: str) -> str:
    return re.sub(r'[\\/:*?"<>|]', "_", name).strip()

def get_pdf_links(soup: BeautifulSoup) -> list:
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/download/document/" in href:
            full_url = urljoin(BASE_URL, href)
            filename = unquote(href.split("/")[-1])
            if not filename.lower().endswith(".pdf"):
                filename = (a.get_text(strip=True) or filename) + ".pdf"
            links.append((filename, full_url))
    return links

def download_pdf(url: str, dest: Path) -> bool:
    if dest.exists():
        print(f"Skip: {dest.name[:65]}")
        return True
    try:
        resp = session.get(url, timeout=60, stream=True)
        resp.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in resp.iter_content(8192):
                f.write(chunk)
        print(f"Downloaded: {dest.name[:65]}")
        return True
    except Exception as e:
        print(f"Error: {dest.name[:50]}: {e}")
        return False

In [6]:
stats = {"total": 0, "ok": 0}

for year, path in PI_PAGES.items():
    url = BASE_URL + path
    print(f"\n{year}  →  {url}")
    try:
        soup = get_soup(url)
    except Exception as e:
        print(f"Site is unavailable: {e}")
        continue

    pdf_links = get_pdf_links(soup)
    print(f"Files found: {len(pdf_links)}")

    year_dir = DATA_DIR / year
    year_dir.mkdir(exist_ok=True)

    for filename, dl_url in pdf_links:
        dest = year_dir / sanitize(filename)
        ok = download_pdf(dl_url, dest)
        stats["total"] += 1
        stats["ok"]    += int(ok)
        time.sleep(0.3)

print(f"\nDownloaded: {stats['ok']}/{stats['total']} files")
print(f"Dir: {DATA_DIR.resolve()}")


2025  →  https://official.satbayev.university/ru/information-telecommunication-technologies/dipomnye-proekty-instituta-avtomatiki-i-informatsionnykh-tekhnologiy/diplomnye-proekty-2025/diplomnye-proekty-pi
Files found: 20
Skip: БАК 2025 10 Городецкая Л.А.pdf
Skip: БАК 2025 11 Искакбаев И.А.pdf
Skip: БАК 2025 12 Калиев, Миргалимов 2.pdf
Skip: БАК 2025 13 Камиль Дильназ.pdf
Skip: БАК 2025 14 Кенжебекқызы П.pdf
Skip: БАК 2025 15 Кенжегулов М.pdf
Skip: БАК 2025 16 Көшкін Әсет Төлегенұлы.pdf
Skip: БАК 2025 17  Кузьмин.pdf
Skip: БАК 2025 18 Қыстақов Е. Н..pdf
Skip: БАК 2025 19 Амангелді_Ернар,_Жансерік_Мирас.pdf
Skip: БАК 2025 20 Тоқтар А.pdf
Skip: БАК 2025 21 Руслан.pdf
Skip: БАК 2025 22 Сон. В.pdf
Skip: БАК 2025 23 Шал Г. Н..pdf
Skip: БАК 2025 3 Абдуали Дінислам, Рақыметқан Ләзат.pdf
Skip: БАК 2025 5 Абдулла К, Араова А.pdf
Skip: БАК 2025 6 Абылкасым Ерарыс.pdf
Skip: БАК 2025 7 Садыкова А.pdf
Skip: БАК 2025 8 Тұрғынбек Бағынұр.pdf
Skip: БАК 2025 9  Эйсмонт В..pdf

2023  →  https://official

In [7]:
all_pdfs = sorted(DATA_DIR.rglob("*.pdf"))
print(f"PDF count in data_PI/: {len(all_pdfs)}\n")
for p in all_pdfs:
    print(f"  {p.relative_to(DATA_DIR)}")

PDF count in data_PI/: 203

  2019\Tursynbek D.Кітаптарды онлайн жалға алуды ұйымдастыратын веб-сайт құру.2019.pdf
  2019\Абдикадирова А.П Косметологияда сараптамалык жуйелерды колдану 2019.pdf
  2019\Ануарбеков А.А Интернет магазин автозапчастей на языке php. 2019 год.pdf
  2019\Аскаров.Е.А.Кол кимылдарын аныктау.2019.pdf
  2019\Ахмедова Г.У. Билимды окыту және бакылау жуйесын азирлеу 2019.pdf
  2019\Байгалиева Г.Е. Андроид платформасында бонустық карталарды сақтауға арналған қосымша.2019.pdf
  2019\Байдулов А.Р. Разработка интернет-магазина с использованием CMS WordPress 2019.pdf
  2019\БалапанТ.Е.Ауыл шаруашылық онлайн жәрмеңкесінің веб-клиентін құру.2019.pdf
  2019\БалгабайБ.М.Kazakh Life мобильді қосымшасын құру.2019.pdf
  2019\Бақытжан Берік.Мейрамхана бизнес кестесінің автоматтандырылған жүйесін әзірлеу.pdf
  2019\Бекболат Ж. Б. Онлайн дәріхана мобилді қосымшасы.2019.pdf
  2019\БекболатЕ.Е.Жеке кабинет үшін аутентификация әдістерін зерттеу және модельдеу.2019.pdf
  2019\Дархан Д